In [1]:
!pip install fastapi uvicorn nest-asyncio

In [2]:
!pip install python-multipart

In [3]:
!pip install ultralytics

# 학습한 모델 웹캠으로 실험

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
import uvicorn
from fastapi.responses import StreamingResponse
import cv2
from ultralytics import YOLO



app = FastAPI()
nest_asyncio.apply()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 모델 가져오기
model = YOLO('./data/best(yolo11m).pt')

# 고정된 파일로 처리할 경우
# camera = cv2.VideoCapture(영상파일경로)

# 웹캠을 연동할 경
camera = cv2.VideoCapture(0)


def gen_frames():
    """웹캠 스트림"""
    while True:
        success, frame = camera.read()

        if not success:
            continue

        #AI모델 호출
        result = model(frame)
        # print(result[0].plot())
        result = result[0].plot()

      
       
        # ✅ 원본 컬러로 전송 (Grayscale 제거)
            ret, buffer = cv2.imencode(".jpg", result)
            frame_bytes = buffer.tobytes()


        yield (
            b"--frame\r\n" b"Content-Type: image/jpeg\r\n\r\n" + frame_bytes + b"\r\n"
        )


@app.get("/video_feed")
def video_feed():
    return StreamingResponse(
        gen_frames(), media_type="multipart/x-mixed-replace; boundary=frame"
    )


config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

In [ ]:
import cv2

# 0번이 안되면 1번으로 바꿔서 시도해보세요
cap = cv2.VideoCapture(0) 
# 윈도우라면 아래 줄 주석을 풀고 시도해보세요
# cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("❌ 카메라를 열 수 없습니다! 번호를 바꾸거나 재부팅하세요.")
else:
    print("✅ 카메라 연결 성공! (q를 누르면 꺼집니다)")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("프레임 읽기 실패...")
            break
            
        cv2.imshow('Test Camera', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
import uvicorn
from fastapi.responses import StreamingResponse
import cv2
from ultralytics import YOLO
import time

# 1. 앱 초기화 (이 부분이 없어서 에러가 났던 겁니다)
app = FastAPI()
nest_asyncio.apply()

# 2. CORS 설정
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 3. 모델 및 카메라 로드
print("모델 로딩 중...")
model = YOLO('./data/best(yolo11m).pt')  # 경로가 맞는지 확인해주세요
print("모델 로드 완료!")

# 카메라 설정 (안 되면 0을 1로 바꿔보세요)
camera = cv2.VideoCapture(0)

def gen_frames():
    """웹캠 스트림 함수 (로직 수정됨)"""
    print("웹캠 스트리밍 시작...") 
    
    while True:
        # 카메라 프레임 읽기
        success, frame = camera.read()

        if not success:
            # 카메라가 안 읽히면 로그를 찍고 잠시 대기
            print("❌ 카메라 프레임 읽기 실패! (카메라 연결 확인 필요)")
            time.sleep(1) 
            continue 

        # AI 모델 감지
        try:
            results = model(frame)
            annotated_frame = results[0].plot() # 감지된 박스 그리기
        except Exception as e:
            print(f"모델 에러: {e}")
            continue

        # 이미지 인코딩
        ret, buffer = cv2.imencode('.jpg', annotated_frame)
        if not ret:
            continue

        frame_bytes = buffer.tobytes()
        
        # 브라우저로 전송
        yield (
            b'--frame\r\n'
            b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n'
        )

@app.get("/video_feed")
def video_feed():
    return StreamingResponse(
        gen_frames(), media_type="multipart/x-mixed-replace; boundary=frame"
    )

# 4. 서버 실행
if __name__ == "__main__":
    print("서버 시작 중... http://127.0.0.1:8000/video_feed 로 접속하세요.")
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

In [ ]:
import torch
print("CUDA 사용 가능 여부:", torch.cuda.is_available())

# 웹캠 성능 끌어올리기 -> OpenVINO 설치해서 프레임드랍 개선하기

In [1]:
!pip install openvino

In [3]:
from ultralytics import YOLO

model = YOLO('./data/best(yolo11m).pt') 
model.export(format='openvino')

Ultralytics 8.3.239  Python-3.13.5 torch-2.9.1+cpu CPU (Intel Core i7-9700 3.00GHz)
YOLO11m summary (fused): 125 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs

PyTorch: starting from 'data\best(yolo11m).pt' with input shape (1, 3, 800, 800) BCHW and output shape(s) (1, 6, 13125) (38.7 MB)

OpenVINO: starting export with openvino 2025.4.1-20426-82bbf0292c5-releases/2025/4...
OpenVINO: export success  5.2s, saved as 'data\best(yolo11m)_openvino_model\' (76.9 MB)

Export complete (6.5s)
Results saved to C:\Users\smhrd\DuDu_Project\data
Predict:         yolo predict task=detect model=data\best(yolo11m)_openvino_model imgsz=800  
Validate:        yolo val task=detect model=data\best(yolo11m)_openvino_model imgsz=800 data=/content/merged_final/data.yaml  
Visualize:       https://netron.app


'data\\best(yolo11m)_openvino_model'

In [5]:
import cv2
from ultralytics import YOLO
from fastapi import FastAPI, Response
from fastapi.responses import StreamingResponse
import uvicorn

# FastAPI 앱 생성
app = FastAPI()

# =========================================================
# [1구역] 전역 설정 (모델 로드 & 함수 정의)
# =========================================================

# 1. 모델 로드 (서버 켜질 때 한 번만 로드)
print("모델 로딩 중...")
model_path = './data/best(yolo11m)_openvino_model/'
model = YOLO(model_path)
print("모델 로딩 완료!")

# 2. 작성자님의 함수들 (DB 저장 등)
def save_log_to_db(cls_id):
    # (작성자님의 DB 코드)
    pass

# =========================================================
# [2구역] 영상 생성기 (Generator)
import time # 시간 지연을 위해 필요

def gen_frames():
    # 1. 카메라 연결 시도
    # 만약 웹캠이 안 되면 0을 1로 바꿔보세요 (cv2.VideoCapture(1))
    camera = cv2.VideoCapture(0)
    
    # 카메라가 제대로 열렸는지 확인
    if not camera.isOpened():
        print("❌ 카메라 장치를 찾을 수 없습니다! (USB 연결 확인)")
        return

    print("🎥 웹캠 스트리밍 시작 (접속 대기 중...)")

    while True:
        # 2. 카메라 프레임 읽기
        success, frame = camera.read()

        if not success:
            # ★ 수정된 부분: 실패하면 미친듯이 로그 찍지 말고 1초 기다림
            # 이렇게 해야 'IOPub data rate' 에러가 안 납니다.
            time.sleep(1) 
            continue 

        # 3. AI 모델 감지
        try:
            results = model(frame, imgsz=800, conf=0.5, verbose=False) # verbose=False로 로그 끄기
            
            # --- [사용자 정의 로직] ---
            for result in results:
                boxes = result.boxes
                if len(boxes) > 0:
                    for box in boxes:
                        cls_id = int(box.cls[0])
                        if cls_id == 1:
                             # save_log_to_db(cls_id) 
                             pass 

            # 4. 화면에 박스 그리기
            annotated_frame = results[0].plot() 
            
        except Exception as e:
            print(f"⚠️ 모델 에러: {e}")
            continue

        # 5. 이미지 인코딩
        ret, buffer = cv2.imencode('.jpg', annotated_frame)
        if not ret:
            continue

        frame_bytes = buffer.tobytes()
        
        # 6. 전송
        yield (
            b'--frame\r\n'
            b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n'
        )
        
    camera.release()


# =========================================================
# [4구역] 라우터 설정 (@app.get)
# =========================================================

@app.get("/video_feed")
def video_feed():
    # 위에서 만든 generate_frames 함수를 실행해서 실시간으로 쏴줍니다.
    return StreamingResponse(gen_frames(), media_type="multipart/x-mixed-replace; boundary=frame")


# [5구역] 서버 실행
# =========================================================
if __name__ == "__main__":
    # 1. 포트 충돌 방지용 설정 (설정 객체 생성)
    import uvicorn
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)

    # 2. 실행 (여기가 다릅니다!)
    # uvicorn.run() 대신 await server.serve()를 씁니다.
    # 이 방식은 주피터 노트북의 실행 흐름(Loop)에 자연스럽게 올라탑니다.
    await server.serve()

모델 로딩 중...
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
모델 로딩 완료!


INFO:     Started server process [22156]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [22156]


In [ ]:
import cv2
import time
from ultralytics import YOLO
from fastapi import FastAPI, Response
from fastapi.responses import StreamingResponse
import uvicorn
import nest_asyncio

app = FastAPI()

# [1구역] 모델 로드
print("모델 로딩 중...")
# 경로가 맞는지 꼭 확인하세요!
model_path = './data/best(yolo11m)_openvino_model/' 
model = YOLO(model_path)
print("✅ 모델 로딩 완료!")

# [2구역] 영상 생성기 (에러 방지 적용됨)
def gen_frames():
    # ★ 카메라 번호: 0이 안 되면 1로 바꿔보세요!
    camera = cv2.VideoCapture(1)
    
    # 카메라 열기 시도
    if not camera.isOpened():
        print("❌ 카메라(0번)를 열 수 없습니다! (번호를 1로 바꿔보세요)")
        return

    print("🎥 웹캠 스트리밍 시작")

    while True:
        success, frame = camera.read()

        if not success:
            # ★ 에러 폭주 방지 (실패하면 1초 쉼)
            print("⚠️ 프레임 읽기 실패 (재시도 중...)")
            time.sleep(1)
            continue 

        try:
            # OpenVINO 가속 추론 (imgsz=800)
            results = model(frame, imgsz=800, conf=0.5, verbose=False)
            annotated_frame = results[0].plot() 
            
        except Exception as e:
            print(f"⚠️ 모델 에러: {e}")
            continue

        # 이미지 인코딩 & 전송
        ret, buffer = cv2.imencode('.jpg', annotated_frame)
        if not ret: continue
        frame_bytes = buffer.tobytes()
        
        yield (b'--frame\r\n'
               b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')
        
    camera.release()

# [3구역] 라우터
@app.get("/video_feed")
def video_feed():
    return StreamingResponse(gen_frames(), media_type="multipart/x-mixed-replace; boundary=frame")

# [4구역] 서버 실행
if __name__ == "__main__":
    nest_asyncio.apply()
    
    # uvicorn 설정 (로그 레벨 조정)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    
    print("🚀 서버 시작: http://localhost:8000/video_feed")
    await server.serve()

모델 로딩 중...
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
✅ 모델 로딩 완료!


INFO:     Started server process [16936]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 서버 시작: http://localhost:8000/video_feed
INFO:     127.0.0.1:57107 - "GET /video_feed HTTP/1.1" 200 OK
❌ 카메라(0번)를 열 수 없습니다! (번호를 1로 바꿔보세요)


## openVino 다시 해보기

In [1]:
from ultralytics import YOLO

model = YOLO('./data/best(yolo11m)(최종).pt') 
model.export(format='openvino')

Ultralytics 8.3.239  Python-3.13.5 torch-2.9.1+cpu CPU (Intel Core i7-9700 3.00GHz)
YOLO11m summary (fused): 125 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs

PyTorch: starting from 'data\best(yolo11m)().pt' with input shape (1, 3, 800, 800) BCHW and output shape(s) (1, 6, 13125) (38.7 MB)

OpenVINO: starting export with openvino 2025.4.1-20426-82bbf0292c5-releases/2025/4...
OpenVINO: export success  5.4s, saved as 'data\best(yolo11m)()_openvino_model\' (76.9 MB)

Export complete (6.6s)
Results saved to C:\Users\smhrd\DuDu_Project\data
Predict:         yolo predict task=detect model=data\best(yolo11m)()_openvino_model imgsz=800  
Validate:        yolo val task=detect model=data\best(yolo11m)()_openvino_model imgsz=800 data=/content/merged_final/data.yaml  
Visualize:       https://netron.app


'data\\best(yolo11m)(최종)_openvino_model'

In [1]:
import cv2
import time
from ultralytics import YOLO
from fastapi import FastAPI, Response
from fastapi.responses import StreamingResponse
import uvicorn
import nest_asyncio

app = FastAPI()

# [1구역] 모델 로드
print("모델 로딩 중...")
# 경로가 맞는지 꼭 확인하세요!
model_path = './data/best(yolo11m)(최종)_openvino_model/' 
model = YOLO(model_path)
print("✅ 모델 로딩 완료!")

# [2구역] 영상 생성기 (에러 방지 적용됨)
def gen_frames():
    # ★ 카메라 번호: 0이 안 되면 1로 바꿔보세요!
    camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    
    # 카메라 열기 시도
    if not camera.isOpened():
        print("❌ 카메라(0번)를 열 수 없습니다! (번호를 1로 바꿔보세요)")
        return

    print("🎥 웹캠 스트리밍 시작")

    while True:
        success, frame = camera.read()

        if not success:
            # ★ 에러 폭주 방지 (실패하면 1초 쉼)
            print("⚠️ 프레임 읽기 실패 (재시도 중...)")
            time.sleep(1)
            continue 

        try:
            # OpenVINO 가속 추론 (imgsz=800)
            results = model(frame, imgsz=800, conf=0.4, verbose=False)
            annotated_frame = results[0].plot() 
            
        except Exception as e:
            print(f"⚠️ 모델 에러: {e}")
            continue

        # 이미지 인코딩 & 전송
        ret, buffer = cv2.imencode('.jpg', annotated_frame)
        if not ret: continue
        frame_bytes = buffer.tobytes()
        
        yield (b'--frame\r\n'
               b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')
        
    camera.release()

# [3구역] 라우터
@app.get("/video_feed")
def video_feed():
    return StreamingResponse(gen_frames(), media_type="multipart/x-mixed-replace; boundary=frame")

# [4구역] 서버 실행
if __name__ == "__main__":
    nest_asyncio.apply()
    
    # uvicorn 설정 (로그 레벨 조정)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    
    print("🚀 서버 시작: http://localhost:8000/video_feed")
    await server.serve()

모델 로딩 중...
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
✅ 모델 로딩 완료!


INFO:     Started server process [11072]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 서버 시작: http://localhost:8000/video_feed
INFO:     127.0.0.1:58851 - "GET /video_feed HTTP/1.1" 200 OK
🎥 웹캠 스트리밍 시작
Loading ./data/best(yolo11m)()_openvino_model/ for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on (CPU)...


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11072]


# 데이터 병합 과정 중에 꼬인 데이터들 다시 재정립하고 시험용으로 n모델로 학습시킨거 웹캠 테스트

In [1]:
import cv2
from ultralytics import YOLO
import time

# ======================================================
# 👇 [설정 영역] 다운로드 받은 모델 경로를 여기에 넣으세요!
# 예: 'C:/Users/내이름/Downloads/best.pt'
# 예시: 앞에 r을 꼭 붙이세요!
model_path = r"C:\Users\smhrd\DuDu_Project\data\best (sDUDU).pt"
# ======================================================

def run_webcam_test():
    # 1. 모델 로드
    print(f"🔄 모델을 불러오는 중... ({model_path})")
    try:
        model = YOLO(model_path)
        print("✅ 모델 로드 성공! 'No Helmet'을 잘 잡는지 확인해봅시다.")
    except Exception as e:
        print(f"❌ 모델을 찾을 수 없습니다. 경로를 확인해주세요!\n에러: {e}")
        return

    # 2. 웹캠 설정 (아까 해결한 윈도우 전용 옵션 적용됨)
    # 0번이 안 되면 1번으로 바꿔보세요.
    # [수정 전]
    # cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    
    # [수정 후] 0을 1로 바꿔보세요!
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    
    # 해상도 설정 (속도 향상을 위해 적당히 조절)
    # cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    # cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    if not cap.isOpened():
        print("❌ 카메라를 열 수 없습니다. 다른 프로그램(Zoom 등)이 켜져 있는지 확인하세요.")
        return

    print("🎥 웹캠 시작! (종료하려면 화면을 클릭하고 'q'를 누르세요)")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("⚠️ 프레임을 읽지 못했습니다. (재시도 중...)")
            time.sleep(1)
            continue

        # 3. 추론 (Inference)
        # conf=0.4: 확신이 40% 이상일 때만 표시
        # imgsz=640: 학습할 때 썼던 사이즈 (일반적으로 640 권장)
        results = model(frame, conf=0.4, imgsz=640, verbose=False)

        # 4. 결과 시각화 (자동으로 박스 그려줌)
        annotated_frame = results[0].plot()

        # 5. 화면 출력
        cv2.imshow("Helmet Detection Test (Press 'q' to exit)", annotated_frame)

        # 'q' 키를 누르면 종료
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 종료 처리
    cap.release()
    cv2.destroyAllWindows()
    print("👋 테스트 종료")

# 실행 함수 호출
if __name__ == "__main__":
    run_webcam_test()

🔄 모델을 불러오는 중... (C:\Users\smhrd\DuDu_Project\data\best (sDUDU).pt)
✅ 모델 로드 성공! 'No Helmet'을 잘 잡는지 확인해봅시다.
🎥 웹캠 시작! (종료하려면 화면을 클릭하고 'q'를 누르세요)


KeyboardInterrupt: 

In [7]:
import os
print("📂 현재 파이썬이 실행 중인 위치:", os.getcwd())
print("파일이 있나요?", os.path.exists('./DuDu_Project/data/best (sDUDU).pt'))

📂 현재 파이썬이 실행 중인 위치: C:\Users\smhrd\DuDu_Project
파일이 있나요? False


### fastAPI 이용해서 웹캠 테스트 시도

In [1]:
import cv2
import time
from ultralytics import YOLO
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import uvicorn
import nest_asyncio

app = FastAPI()

# 1. 모델 경로 (아까 성공한 경로 그대로!)
# r을 붙여서 절대 경로로 적어주세요
model_path = r"C:\Users\smhrd\DuDu_Project\data\best (sDUDU).pt"
print(f"🔄 모델 로딩 중... ({model_path})")
try:
    model = YOLO(model_path)
    print("✅ 모델 로딩 완료!")
except Exception as e:
    print(f"❌ 모델을 찾을 수 없습니다: {e}")

# 2. 영상 생성기 (수리 완료됨)
def gen_frames():
    # 0번이 안 되면 1번으로! (아까 성공했던 번호 쓰세요)
    camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    
    # ★ 핵심 수정: 해상도 설정 코드 삭제 (주석 처리) ★
    # camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    # camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    if not camera.isOpened():
        print("❌ 카메라를 열 수 없습니다.")
        return

    print("🎥 웹캠 스트리밍 시작")

    while True:
        success, frame = camera.read()
        if not success:
            # 실패해도 죽지 않고 기다림
            time.sleep(0.1)
            continue

        try:
            # 추론
            results = model(frame, conf=0.4, imgsz=640, verbose=False)
            annotated_frame = results[0].plot()
        except Exception as e:
            print(f"⚠️ 모델 에러: {e}")
            continue

        # 인코딩
        ret, buffer = cv2.imencode('.jpg', annotated_frame)
        if not ret: continue
        frame_bytes = buffer.tobytes()
        
        yield (b'--frame\r\n'
               b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')

# 3. 라우터
@app.get("/video_feed")
def video_feed():
    return StreamingResponse(gen_frames(), media_type="multipart/x-mixed-replace; boundary=frame")

# [4구역] 서버 실행
if __name__ == "__main__":
    import uvicorn
    
    # nest_asyncio는 주피터 노트북의 이벤트 루프 충돌을 막아줍니다 (그대로 유지)
    nest_asyncio.apply()

    # ⚠️ 수정된 부분: uvicorn.run() 대신 Config와 Server 객체를 사용합니다.
    print("🚀 서버 시작: http://localhost:8000/video_feed 로 접속하세요!")
    
    # 1. 설정(Config) 객체 생성
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    
    # 2. 서버(Server) 객체 생성
    server = uvicorn.Server(config)
    
    # 3. 비동기 실행 (await를 붙여야 주피터 루프와 충돌하지 않음)
    await server.serve()

🔄 모델 로딩 중... (C:\Users\smhrd\DuDu_Project\data\best (sDUDU).pt)
✅ 모델 로딩 완료!
🚀 서버 시작: http://localhost:8000/video_feed 로 접속하세요!


INFO:     Started server process [2672]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:60051 - "GET /video_feed HTTP/1.1" 200 OK
🎥 웹캠 스트리밍 시작


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [2672]
